# 부록 01: RunPod와 외부 접속 환경 준비

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">RunPod에서 모델을 실행하고, 내 PC에서 API로 호출할 환경을 준비합니다. 준비를 마치면 교안 01에서 모델 불러오기·서버 실행·API 요청을 진행합니다.</p>
</div>

## 1. Pod 생성 설정

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><a style="color:#245fa0;text-decoration:underline;" href="https://console.runpod.io/deploy">Deploy a Pod</a> 화면을 엽니다. Workload에서 CUDA·포트 설정 → Compute에서 GPU 선택 → Storage에서 디스크 설정 → Deploy Pod 순서로 진행합니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">선택 기준</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">GPU</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">NVIDIA GPU, VRAM 16GB 이상</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">권장 GPU</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">RTX A4000·RTX 2000 Ada 16GB / RTX 3090·RTX A5000 24GB</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">시스템 RAM</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">16GB 이상</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">디스크</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">80GB 이상 — 원본·양자화·변환 모델 보관</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">템플릿</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">PyTorch → CUDA 13.0.0</td>
</tr>
</tbody>
</table>
</div>

### 1-1. Workload: CUDA 13 템플릿 선택

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">Workload 영역에서 공식 PyTorch 템플릿을 고르고, 오른쪽 CUDA 버전 선택 메뉴에서 CUDA 13.0.0을 선택합니다.</p>
<p style="margin:6px 0;">CUDA 13.0 선택 이유: <a style="color:#245fa0;text-decoration:underline;" href="https://github.com/vllm-project/vllm/blob/v0.27.0/docker/Dockerfile#L22">vLLM 0.27.0의 공식 Dockerfile</a>이 CUDA 13.0.3을 기준으로 하므로, RunPod에서도 CUDA 13.0 템플릿을 선택합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/screenshots/01_workload_buttons.png" width="1050" alt="Workload 전체에서 CUDA 버전 선택 메뉴와 오른쪽 Set overrides 버튼의 위치를 확인합니다.">

### 1-2. Workload: HTTP 8000 포트 설정

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">설정 목적: 내 PC에서 vLLM API에 접속하도록 HTTP 8000 포트를 공개합니다. 기존 8888은 Jupyter, 22는 SSH 접속용으로 유지합니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">바로 위 화면의 Set overrides 버튼을 눌러 Edit template overrides 창을 엽니다.</li>
<li style="margin:3px 0;">Exposed HTTP ports의 기존 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">8888</code>을 유지하고 빈 칸에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">8000</code>을 추가합니다. Exposed TCP ports의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">22</code>도 유지합니다.</li>
<li style="margin:3px 0;">창 맨 아래의 Set overrides 버튼을 눌러 설정을 적용합니다.</li>
</ol>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/screenshots/02_overrides_ports_apply.png" alt="Edit template overrides 창 전체에서 HTTP 포트 입력란과 하단 Set overrides 적용 버튼을 확인합니다." width="680">

### 1-3. Compute: GPU 1장 선택

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">설정 목적: 모델은 한 번에 하나씩 실행하므로 GPU 1장을 선택합니다. On-Demand는 Pod 실행 시간에 따라 GPU 사용 요금을 내는 방식입니다.</p>
<p style="margin:6px 0;">GPU → Available을 선택하고 정렬 메뉴를 Price로 바꿉니다. 옆의 정렬 방향 버튼으로 낮은 가격순으로 맞춘 뒤 표의 권장 GPU 중 VRAM 16GB·RAM 16GB 이상인 후보를 선택합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/screenshots/03_compute_selection.png" width="1050" alt="Compute 전체에서 Available 탭, Price 메뉴와 정렬 방향 버튼을 확인합니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 GPU count에서 1, Instance pricing에서 On-Demand를 선택합니다. GPU 이름·가격·재고는 바뀌므로 표의 최소 조건을 기준으로 고릅니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/screenshots/04_gpu_count_pricing.png" width="1050" alt="GPU count의 1 버튼과 Instance pricing의 On-Demand 선택 위치를 확인합니다.">

### 1-4. Storage 설정과 Pod 생성

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">설정 목적: 모델 파일과 과정 패키지를 저장할 공간을 확보합니다. Storage → Container disk를 80GB 이상으로 설정합니다.</p>
<p style="margin:6px 0;">오른쪽 요약에서 GPU 1장·디스크 80GB·시간당 총요금을 확인한 뒤 Deploy Pod 버튼을 누릅니다. 컨테이너 준비가 끝나면 과정 파일을 복사합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/screenshots/05_storage_deploy.png" width="1050" alt="Storage의 Container disk 입력란과 오른쪽 요약 카드 하단의 Deploy Pod 버튼을 확인합니다.">

## 2. 외부 접속 주소 확인과 내 PC에 저장

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">설정 목적: 내 PC의 클라이언트가 사용할 vLLM 서버의 공개 주소를 확인합니다.</p>
<p style="margin:6px 0;">Pods에서 내 Pod를 열고 Connect 탭을 선택합니다. Port 8000 행의 HTTP Service 링크를 오른쪽 클릭 → 링크 주소 복사합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/screenshots/07_connection_url_location.png" width="1050" alt="RunPod의 Connect 탭에서 공개 URL을 복사할 Port 8000의 HTTP Service 링크를 빨간 테두리로 표시한 실제 캡처입니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">누를 곳: Pods → 내 Pod → Connect → 빨간 테두리의 HTTP Service를 오른쪽 클릭 → 링크 주소 복사.</p>
<p style="margin:6px 0;">콘솔 위치: <a style="color:#245fa0;text-decoration:underline;" href="https://console.runpod.io/pods">RunPod Pods</a>.</p>
<p style="margin:6px 0;">내 PC의 VS Code에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>를 엽니다. 교안이 들어 있는 단원 폴더의 한 단계 위에 있는 파일이며, 없으면 만듭니다. 기존 설정은 유지하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_SERVER_URL</code>을 추가하거나 수정하되, 아래 예시 주소를 방금 복사한 자신의 HTTP Service 링크로 바꿉니다.</p>
</div>

```dotenv
VLLM_SERVER_URL=https://YOUR-POD-ID-8000.proxy.runpod.net
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">주소에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code>은 붙이지 않습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code>은 교안의 Python 클라이언트가 붙입니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">모델 API는 교안 01에서 서버를 실행하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>를 확인한 뒤 사용합니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">용도</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">HTTP <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">8888</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">Jupyter 접속</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">HTTP <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">8000</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">교안의 vLLM API</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">공개 주소</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://&lt;Pod ID&gt;-8000.proxy.runpod.net</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><a style="color:#245fa0;text-decoration:underline;" href="https://docs.runpod.io/pods/configuration/expose-ports">RunPod 포트 공개 안내</a></p>
</div>

## 3. API 키와 내 PC의 접속 설정

### 3-1. RunPod: API 키 생성

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료</code> 폴더에서 새 터미널을 열고 아래 명령을 실행합니다. API 키는 고정 접두사 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enkoa-vllm-</code> 뒤에 무작위 문자열을 붙여 생성합니다. 이 키는 직접 실행할 vLLM 서버의 요청 인증에 사용하며 RunPod 계정 API 키와는 별개입니다.</p>
</div>

```bash
cd day75_vLLM_추론엔진_서빙
mkdir -p output
VLLM_API_KEY="$(uv run python -c 'import secrets; print("enkoa-vllm-" + secrets.token_urlsafe(32))')" &&
export VLLM_API_KEY &&
printf 'VLLM_API_KEY=%s\n' "$VLLM_API_KEY"
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안의 서버 실행 명령에 있는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key &quot;$VLLM_API_KEY&quot;</code>가 생성한 키를 vLLM의 인증키로 등록합니다. 내 PC의 클라이언트는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 읽은 같은 키를 요청에 보내며, vLLM은 키가 일치하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> API 요청을 허용합니다. 키를 알고 있는 사람이나 프로그램은 모두 사용할 수 있으며, 개인별 신원을 구분하는 키는 아닙니다.</p>
<p style="margin:6px 0;">여러 API 키를 허용하려면 서버 실행 시 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key</code> 뒤에 키들을 함께 지정합니다. 요청은 등록한 키 중 하나와 일치하면 통과하므로, 클라이언트마다 다른 키를 주거나 교체 중에 기존 키와 새 키를 함께 둘 수 있습니다. vLLM은 키별 사용자 계정이나 사용량을 관리하지 않습니다.</p>
<p style="margin:6px 0;">출력된 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY=...</code> 한 줄 전체를 다음 절에서 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 복사합니다. 이 RunPod 터미널은 교안 01의 서버 실행에도 계속 사용합니다.</p>
</div>

### 3-2. 내 PC: API 키 저장과 노트북 커널 준비

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">2절에서 URL을 저장한 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에 3-1절에서 출력한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY=...</code> 한 줄 전체를 추가합니다. 같은 이름의 항목이 있으면 값을 수정합니다. 저장을 마치면 아래 두 항목이 있어야 합니다.</p>
</div>

```dotenv
VLLM_SERVER_URL=https://YOUR-POD-ID-8000.proxy.runpod.net
VLLM_API_KEY=PASTE_GENERATED_KEY
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">첫째 줄은 2절에서 저장한 자신의 URL을 유지합니다. 둘째 줄은 3-1절에서 출력한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY=...</code> 한 줄 전체로 교체합니다.</p>
<p style="margin:6px 0;">내 PC의 VS Code에서 과정 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.venv</code>를 노트북 커널로 선택합니다. 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 수정했다면 내 PC의 VS Code에서 실행 중인 노트북 커널을 다시 시작하고, 교안의 환경변수를 읽는 준비 셀부터 다시 실행합니다.</p>
<p style="margin:6px 0;">키 생성 명령을 다시 실행하면 키가 바뀝니다. 그때는 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>도 새 출력으로 갱신하고 위의 커널 재시작 절차를 진행합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/07_runpod_external_api.png" width="1050" alt="내 PC의 노트북이 .env에 저장한 공개 HTTPS 주소와 API 키로 RunPod의 vLLM 서버를 호출합니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">준비 완료: 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에 공개 주소와 API 키를 저장했고, RunPod에는 같은 키가 환경변수로 설정되어 있습니다.</p>
<p style="margin:6px 0;">이제 <a style="color:#245fa0;text-decoration:underline;" href="%EA%B5%90%EC%95%88_01_%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_LLM_API_%EC%84%9C%EB%B9%99.ipynb">교안 01: 오픈소스 LLM을 API로 서비스하기</a>에서 모델 준비부터 진행합니다.</p>
</div>

## 4. RunPod 서버 환경

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">저장소 루트의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pyproject.toml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code>이 PC와 RunPod에서 함께 쓰는 공용 Python 환경을 관리합니다. Linux x86_64에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm 0.27.0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">torch 2.13.0+cu130</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">torchvision 0.28.0+cu130</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">triton 3.7.1</code>과 필요한 의존 패키지를 잠금 파일에 고정합니다. vLLM은 Linux에서만 설치되며, PC의 다른 운영체제는 해당 운영체제용 PyTorch 휠을 사용합니다.</p>
<p style="margin:6px 0;">교안의 서버 명령은 day75 폴더에서 실행해도 상위 저장소의 공용 uv 환경을 찾아 사용합니다. 저장소의 일반적인 환경 동기화에 vLLM이 포함되므로 별도 설치 명령은 필요하지 않습니다.</p>
<p style="margin:6px 0;">버전 기준은 <a style="color:#245fa0;text-decoration:underline;" href="https://docs.vllm.ai/en/v0.27.0/getting_started/installation/gpu/">vLLM 0.27.0 공식 설치 안내</a>와 <a style="color:#245fa0;text-decoration:underline;" href="https://github.com/vllm-project/vllm/releases/tag/v0.27.0">릴리스 노트</a>를 참고합니다.</p>
</div>